# sweep_objective_settings notebook

This notebook embeds the full code from `sweep_objective_settings.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

import argparse
import subprocess
from pathlib import Path
from typing import List

import pandas as pd


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Run calibration sweep across KGE mode and inverse-flow offsets."
    )
    parser.add_argument("--basins", required=True, help="Comma-separated basin IDs")
    parser.add_argument("--kge_modes", default="kge2009,kge2012")
    parser.add_argument("--inv_offsets", default="1.5")
    parser.add_argument("--max_gens", type=int, default=20)
    parser.add_argument("--np_max", type=int, default=28)
    parser.add_argument("--np_min", type=int, default=8)
    parser.add_argument("--n_days_total", type=int, default=4018)
    parser.add_argument("--warm_span_days", type=int, default=365)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--objective_target",
        choices=[
            "full",
            "invq_only",
            "lognse_only",
            "logkge_only",
            "q_invq_balanced",
            "kge_bfi_only",
            "bfi_q_invq_triple",
            "bfi_q_logkge_triple",
        ],
        default="invq_only",
    )
    parser.add_argument("--lognse_offset", type=float, default=1e-2)
    parser.add_argument("--lh_alpha", type=float, default=0.925)
    parser.add_argument("--lh_passes", type=int, default=3)
    parser.add_argument("--sm_mode", choices=["fixed", "vary"], default="vary")
    parser.add_argument("--sm_rel_range", type=float, default=0.20)
    parser.add_argument("--sm_bounds_mode", choices=["relative", "absolute"], default="absolute")
    parser.add_argument("--sm_abs_min", type=float, default=0.0)
    parser.add_argument("--sm_abs_max", type=float, default=8000.0)
    parser.add_argument("--q_obs_unit", choices=["m3s", "mmday"], default="m3s")
    parser.add_argument("--area_field", default="area_estre")
    parser.add_argument(
        "--output_csv",
        default="Results/calibration/objective_sweep_results.csv",
    )
    args = parser.parse_known_args()[0]

    basins = args.basins
    kge_modes: List[str] = [m.strip() for m in args.kge_modes.split(",") if m.strip()]
    offsets: List[float] = [float(x.strip()) for x in args.inv_offsets.split(",") if x.strip()]

    tmp_dir = Path("Results/calibration")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for mode in kge_modes:
        for offset in offsets:
            out_path = tmp_dir / f"_tmp_calib_{mode}_c{str(offset).replace('.', 'p')}.csv"
            cmd = [
                "python",
                "calibrate_validate_dpwm.py",
                "--basins",
                basins,
                "--kge_mode",
                mode,
                "--inv_flow_offset",
                str(offset),
                "--max_gens",
                str(args.max_gens),
                "--np_max",
                str(args.np_max),
                "--np_min",
                str(args.np_min),
                "--n_days_total",
                str(args.n_days_total),
                "--warm_span_days",
                str(args.warm_span_days),
                "--seed",
                str(args.seed),
                "--objective_target",
                str(args.objective_target),
                "--lognse_offset",
                str(args.lognse_offset),
                "--lh_alpha",
                str(args.lh_alpha),
                "--lh_passes",
                str(args.lh_passes),
                "--sm_mode",
                str(args.sm_mode),
                "--sm_rel_range",
                str(args.sm_rel_range),
                "--sm_bounds_mode",
                str(args.sm_bounds_mode),
                "--sm_abs_min",
                str(args.sm_abs_min),
                "--sm_abs_max",
                str(args.sm_abs_max),
                "--q_obs_unit",
                str(args.q_obs_unit),
                "--area_field",
                str(args.area_field),
                "--output_csv",
                str(out_path),
            ]
            print("Running:", " ".join(cmd))
            subprocess.run(cmd, check=True)

            df = pd.read_csv(out_path)
            df["kge_mode"] = mode
            df["inv_flow_offset"] = float(offset)
            df["objective_target"] = str(args.objective_target)
            df["lognse_offset"] = float(args.lognse_offset)
            df["lh_alpha"] = float(args.lh_alpha)
            df["lh_passes"] = int(args.lh_passes)
            df["sm_mode"] = str(args.sm_mode)
            df["sm_rel_range"] = float(args.sm_rel_range)
            df["sm_bounds_mode"] = str(args.sm_bounds_mode)
            df["sm_abs_min"] = float(args.sm_abs_min)
            df["sm_abs_max"] = float(args.sm_abs_max)
            df["q_obs_unit"] = str(args.q_obs_unit)
            df["area_field"] = str(args.area_field)
            rows.append(df)

    all_df = pd.concat(rows, ignore_index=True)
    all_df = all_df.sort_values(
        by=["basin_id", "F_cal", "F_val"], ascending=[True, False, False]
    ).reset_index(drop=True)
    Path(args.output_csv).parent.mkdir(parents=True, exist_ok=True)
    all_df.to_csv(args.output_csv, index=False)
    print(f"Wrote sweep table: {args.output_csv}")

    best = (
        all_df.sort_values(by=["basin_id", "F_cal", "F_val"], ascending=[True, False, False])
        .groupby("basin_id", as_index=False)
        .first()
    )
    best_out = Path(args.output_csv).with_name(
        Path(args.output_csv).stem + "_best_per_basin.csv"
    )
    best.to_csv(best_out, index=False)
    print(f"Wrote best-per-basin table: {best_out}")
    print(best[["basin_id", "kge_mode", "inv_flow_offset", "F_cal", "F_val"]])


# Execute the script entry point
main()
